# Portal Method – Lateral Load Analysis
## Approximate Analysis of Multi-Storey Frames

This notebook applies the **Portal Method** to approximate the shear forces and
bending moments in beams and columns of a multi-storey, multi-bay rectangular
frame under lateral (wind / seismic) loads.

### Assumptions of the Portal Method
1. A point of contra-flexure (inflection point) exists at the **mid-height** of every column.
2. A point of contra-flexure exists at the **mid-span** of every beam.
3. Interior columns carry **twice** the shear of exterior columns in the same storey.

### How to use
1. Fill in the values in `portal.xlsx` (see the `data/` folder for the template).
2. Run all cells top to bottom (`Runtime → Run all` in Colab, or `Kernel → Restart & Run All` locally).
3. The shear-force and bending-moment diagrams are plotted at the end.


In [ ]:
# Standard library imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Display settings
pd.set_option("display.float_format", "{:.3f}".format)
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})


## 1. Load Input Data

In [ ]:
# ── USER: change this path if needed ──────────────────────────────────────────
EXCEL_PATH = "data/portal.xlsx"
# ─────────────────────────────────────────────────────────────────────────────

REQUIRED_COLS = [
    "beams",
    "force",
    "distance between beams from top",
    "distance between columns",
]

df = pd.read_excel(EXCEL_PATH, usecols=REQUIRED_COLS)

print("Input data loaded successfully:")
print(df.to_string(index=False))


## 2. Derived Geometry & Cumulative Loads

In [ ]:
# Cumulative storey shear (sum of lateral forces from top down)
df["cumulative_shear"] = df["force"].cumsum()

# Cumulative co-ordinates (for plotting)
df["x_col"]  = df["distance between columns"].cumsum()       # right edge of each bay
df["y_beam"] = df["distance between beams from top"].cumsum() # elevation measured from top

# Frame dimensions
m = int(df["distance between beams from top"].count())   # number of storeys
n = int(df["distance between columns"].count()) + 1      # number of column lines

total_height = float(df["distance between beams from top"].sum())

print(f"Number of storeys       (m) : {m}")
print(f"Number of column lines  (n) : {n}")
print(f"Total frame height          : {total_height} m")
print()
print("Updated DataFrame:")
print(df.to_string(index=False))


## 3. Column Shear Forces  `q`

By the Portal Method the total storey shear *V* is distributed as:
- **Exterior columns**: `q_ext = V / (2*(n-2) + 2)`
- **Interior columns**: `q_int = 2 * q_ext`


In [ ]:
# q[i][j] = shear in column j of storey i  (i=0 → top storey)
q = [[0.0] * n for _ in range(m)]

for i in range(m):
    V = float(df.iloc[i]["cumulative_shear"])
    q_ext = V / (2 * (n - 2) + 2)          # exterior column shear
    q[i][0]     = q_ext                      # left exterior
    q[i][n - 1] = q_ext                      # right exterior
    for j in range(1, n - 1):               # interior columns get double
        q[i][j] = 2 * q_ext

# Pretty-print
print("Column Shear Forces  q[storey][column]  (kN)")
header = "           " + "".join(f"  Col-{j+1:>2d}" for j in range(n))
print(header)
for i in range(m):
    row = f"  Storey {i+1:>2d} " + "".join(f"  {q[i][j]:6.2f}" for j in range(n))
    print(row)


## 4. Beam Shear Forces  `v`

At each beam-column joint, moment equilibrium is applied.
Column end-moment = `q * h/2`.  Beam shears are found working left-to-right
across each storey.


In [ ]:
# v[i][j] = shear in beam j (bay j) of storey i
v = [[0.0] * (n - 1) for _ in range(m)]

for i in range(m):
    h_curr = float(df.iloc[i]["distance between beams from top"])

    for j in range(n - 1):
        # Sum of column moments at this joint
        M_col_below = q[i][j] * (h_curr / 2)
        M_col_above = 0.0
        if i > 0:
            h_above     = float(df.iloc[i - 1]["distance between beams from top"])
            M_col_above = q[i - 1][j] * (h_above / 2)

        total_M_col = M_col_above + M_col_below
        L_beam      = float(df.iloc[j]["distance between columns"])

        if j == 0:
            # First bay: all column moment is balanced by this beam alone
            v[i][j] = total_M_col / (L_beam / 2)
        else:
            # Subsequent bays: subtract the moment already taken by the beam to the left
            L_beam_prev  = float(df.iloc[j - 1]["distance between columns"])
            M_beam_left  = v[i][j - 1] * (L_beam_prev / 2)
            v[i][j]      = (total_M_col - M_beam_left) / (L_beam / 2)

print("Beam Shear Forces  v[storey][bay]  (kN)")
header = "           " + "".join(f"   Bay-{j+1:>2d}" for j in range(n - 1))
print(header)
for i in range(m):
    row = f"  Storey {i+1:>2d} " + "".join(f"   {v[i][j]:6.2f}" for j in range(n - 1))
    print(row)


## 5. Bending Moments

In [ ]:
# Column end-moments: M = q * h/2
qm = [
    [q[i][j] * float(df.iloc[i]["distance between beams from top"]) / 2
     for j in range(n)]
    for i in range(m)
]

# Beam end-moments: M = v * L/2
vm = [
    [v[i][j] * float(df.iloc[j]["distance between columns"]) / 2
     for j in range(n - 1)]
    for i in range(m)
]

print("Column Bending Moments  qm[storey][column]  (kN·m)")
header = "           " + "".join(f"  Col-{j+1:>2d}" for j in range(n))
print(header)
for i in range(m):
    row = f"  Storey {i+1:>2d} " + "".join(f"  {qm[i][j]:6.2f}" for j in range(n))
    print(row)

print()
print("Beam Bending Moments  vm[storey][bay]  (kN·m)")
header = "           " + "".join(f"   Bay-{j+1:>2d}" for j in range(n - 1))
print(header)
for i in range(m):
    row = f"  Storey {i+1:>2d} " + "".join(f"   {vm[i][j]:6.2f}" for j in range(n - 1))
    print(row)


## 6. Shear Force Diagram

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
ax.set_aspect("equal")
ax.set_title("Shear Force Diagram – Portal Method", fontsize=13, fontweight="bold")
ax.set_xlabel("Horizontal distance (m)")
ax.set_ylabel("Height (m)")
ax.grid(True, linestyle="--", alpha=0.3)

# ── Frame skeleton ────────────────────────────────────────────────────────────
for j in range(n):
    col_x = 0.0 if j == 0 else float(df.iloc[j - 1]["x_col"])
    ax.plot([col_x, col_x], [0, total_height], color="black", linewidth=2, zorder=2)

for i in range(m):
    y_beam = total_height if i == 0 else total_height - float(df.iloc[i - 1]["y_beam"])
    ax.plot([0, float(df["distance between columns"].sum())],
            [y_beam, y_beam], color="black", linewidth=2, zorder=2)

# ── Column shear blocks ───────────────────────────────────────────────────────
q_max         = max(max(row) for row in q)
min_col_span  = float(df["distance between columns"].min())
scale_q       = 0.35 * min_col_span / q_max if q_max else 0.05

for i in range(m):
    y2 = total_height if i == 0 else total_height - float(df.iloc[i - 1]["y_beam"])
    y1 = 0.0          if i == m - 1 else total_height - float(df.iloc[i]["y_beam"])
    for j in range(n):
        col_x = 0.0 if j == 0 else float(df.iloc[j - 1]["x_col"])
        sv    = q[i][j]
        ox    = col_x - sv * scale_q
        ax.fill([col_x, ox, ox, col_x], [y1, y1, y2, y2], color="steelblue", alpha=0.25)
        ax.plot([ox, ox], [y1, y2], color="steelblue", lw=1.5)
        ax.plot([col_x, ox], [y1, y1], color="steelblue", lw=1.5)
        ax.plot([col_x, ox], [y2, y2], color="steelblue", lw=1.5)
        ax.text(ox - 0.15, (y1 + y2) / 2, f"{sv:.2f}",
                color="navy", fontsize=9, ha="right",
                bbox=dict(facecolor="white", edgecolor="none", alpha=0.7, pad=1))

# ── Beam shear blocks ─────────────────────────────────────────────────────────
v_max          = max(max(row) for row in v)
min_storey_h   = float(df["distance between beams from top"].min())
scale_v        = 0.35 * min_storey_h / v_max if v_max else 0.05

for i in range(m):
    y_beam = total_height if i == 0 else total_height - float(df.iloc[i - 1]["y_beam"])
    for j in range(n - 1):
        x1 = 0.0 if j == 0 else float(df.iloc[j - 1]["x_col"])
        x2 = float(df.iloc[j]["x_col"])
        sv = v[i][j]
        oy = y_beam + sv * scale_v
        ax.fill([x1, x2, x2, x1], [y_beam, y_beam, oy, oy], color="tomato", alpha=0.25)
        ax.plot([x1, x2], [oy, oy], color="tomato", lw=1.5)
        ax.plot([x1, x1], [y_beam, oy], color="tomato", lw=1.5)
        ax.plot([x2, x2], [y_beam, oy], color="tomato", lw=1.5)
        ax.text((x1 + x2) / 2, oy + 0.1, f"{sv:.2f}",
                color="darkred", fontsize=9, ha="center",
                bbox=dict(facecolor="white", edgecolor="none", alpha=0.7, pad=1))

col_patch  = mpatches.Patch(color="steelblue", alpha=0.5, label="Column shear (kN)")
beam_patch = mpatches.Patch(color="tomato",    alpha=0.5, label="Beam shear (kN)")
ax.legend(handles=[col_patch, beam_patch], loc="lower right")
plt.tight_layout()
plt.savefig("shear_force_diagram.png", dpi=150, bbox_inches="tight")
plt.show()


## 7. Bending Moment Diagram

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
ax.set_aspect("equal")
ax.set_title("Bending Moment Diagram – Portal Method", fontsize=13, fontweight="bold")
ax.set_xlabel("Horizontal distance (m)")
ax.set_ylabel("Height (m)")
ax.grid(True, linestyle="--", alpha=0.3)

# ── Frame skeleton ────────────────────────────────────────────────────────────
for j in range(n):
    col_x = 0.0 if j == 0 else float(df.iloc[j - 1]["x_col"])
    ax.plot([col_x, col_x], [0, total_height], color="black", linewidth=2, zorder=2)

for i in range(m):
    y_beam = total_height if i == 0 else total_height - float(df.iloc[i - 1]["y_beam"])
    ax.plot([0, float(df["distance between columns"].sum())],
            [y_beam, y_beam], color="black", linewidth=2, zorder=2)

# ── Column moment diagram ─────────────────────────────────────────────────────
qm_max        = max(max(row) for row in qm)
min_col_span  = float(df["distance between columns"].min())
scale_qm      = 0.35 * min_col_span / qm_max if qm_max else 0.05

for i in range(m):
    y2 = total_height if i == 0 else total_height - float(df.iloc[i - 1]["y_beam"])
    y1 = 0.0          if i == m - 1 else total_height - float(df.iloc[i]["y_beam"])
    for j in range(n):
        col_x  = 0.0 if j == 0 else float(df.iloc[j - 1]["x_col"])
        mv     = qm[i][j]
        offset = mv * scale_qm
        x_top  = col_x + offset
        x_bot  = col_x - offset
        ax.fill([col_x, x_bot, x_top, col_x], [y1, y1, y2, y2], color="mediumpurple", alpha=0.25)
        ax.plot([x_bot, x_top], [y1, y2], color="mediumpurple", lw=1.5)
        ax.plot([col_x, x_bot], [y1, y1], color="mediumpurple", lw=1.5)
        ax.plot([col_x, x_top], [y2, y2], color="mediumpurple", lw=1.5)
        ax.text(x_top + 0.1, y2 - 0.3, f"{mv:.2f}",
                color="indigo", fontsize=9, fontweight="bold",
                bbox=dict(facecolor="white", edgecolor="none", alpha=0.7, pad=1))

# ── Beam moment diagram ───────────────────────────────────────────────────────
vm_max       = max(max(row) for row in vm)
min_storey_h = float(df["distance between beams from top"].min())
scale_vm     = 0.35 * min_storey_h / vm_max if vm_max else 0.05

for i in range(m):
    y_beam = total_height if i == 0 else total_height - float(df.iloc[i - 1]["y_beam"])
    for j in range(n - 1):
        x1     = 0.0 if j == 0 else float(df.iloc[j - 1]["x_col"])
        x2     = float(df.iloc[j]["x_col"])
        mv     = vm[i][j]
        x_mid  = (x1 + x2) / 2
        y_left  = y_beam + mv * scale_vm
        y_right = y_beam - mv * scale_vm
        ax.fill([x1, x1, x_mid], [y_beam, y_left, y_beam], color="coral", alpha=0.2)
        ax.fill([x_mid, x2, x2], [y_beam, y_right, y_beam], color="coral", alpha=0.2)
        ax.plot([x1, x2], [y_left, y_right], color="coral", lw=1.5)
        ax.plot([x1, x1], [y_beam, y_left], color="coral", lw=1.5)
        ax.plot([x2, x2], [y_beam, y_right], color="coral", lw=1.5)
        ax.text(x1 + 0.2, y_left + 0.1, f"{mv:.2f}",
                color="darkred", fontsize=9, fontweight="bold",
                bbox=dict(facecolor="white", edgecolor="none", alpha=0.7, pad=1))

col_patch  = mpatches.Patch(color="mediumpurple", alpha=0.5, label="Column moment (kN·m)")
beam_patch = mpatches.Patch(color="coral",        alpha=0.5, label="Beam moment (kN·m)")
ax.legend(handles=[col_patch, beam_patch], loc="lower right")
plt.tight_layout()
plt.savefig("bending_moment_diagram.png", dpi=150, bbox_inches="tight")
plt.show()


## 8. Summary Tables

In [ ]:
import itertools

# Column results
col_records = [
    {"Storey": i + 1, "Column": j + 1,
     "Shear (kN)": round(q[i][j], 3),
     "Moment (kN·m)": round(qm[i][j], 3)}
    for i, j in itertools.product(range(m), range(n))
]

# Beam results
beam_records = [
    {"Storey": i + 1, "Bay": j + 1,
     "Shear (kN)": round(v[i][j], 3),
     "Moment (kN·m)": round(vm[i][j], 3)}
    for i, j in itertools.product(range(m), range(n - 1))
]

print("=== Column Results ===")
print(pd.DataFrame(col_records).to_string(index=False))
print()
print("=== Beam Results ===")
print(pd.DataFrame(beam_records).to_string(index=False))
